In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score


# -------------------------------------------------------------------------
# Data structures
# -------------------------------------------------------------------------

@dataclass
class DSLMetaModel:
    """Container for a per-frame DSL meta-model."""
    frame: str
    model: Any  # e.g., sklearn Pipeline
    surrogate_cols: List[str]
    extra_feature_cols: List[str]


@dataclass
class DSLFrameResult:
    """
    Results for a single frame:
      - theta_hat_mean_prevalence: DSL estimate of mean Y over the target corpus
      - meta_model: DSLMetaModel with the fitted meta learner
      - p_all: per-document ensemble probabilities for all docs
      - p_labeled: ensemble probabilities for labeled docs (aligned with y_labeled)
      - thresholds: (optional) tuned classification threshold for this frame
    """
    frame: str
    theta_hat_mean_prevalence: float
    meta_model: DSLMetaModel
    p_all: np.ndarray
    p_labeled: np.ndarray
    threshold: Optional[float] = None
    ci_lower: Optional[float] = None      
    ci_upper: Optional[float] = None      


# -------------------------------------------------------------------------
# Surrogate matrix construction
# -------------------------------------------------------------------------

def build_surrogate_df(
    frames: List[str],
    prob_surrogates: Dict[str, Dict[str, np.ndarray]],
    label_surrogates: Optional[Dict[str, pd.DataFrame]] = None,
) -> pd.DataFrame:
    """
    Build a surrogate feature matrix S for a set of documents.

    Parameters
    ----------
    frames : list of str
        Frame names (e.g., 7 frame columns).
    prob_surrogates : dict
        Mapping from surrogate name -> dict(frame -> 1D np.array of probs).
        Example:
            prob_surrogates = {
                "NB": nb_res["val_continuous"],        # val_continuous[frame] -> np.array
                "BERT": bert_res["val_continuous"],
                "DeBERTa": deberta_res["val_continuous"],
            }
    label_surrogates : dict, optional
        Mapping from surrogate name -> DataFrame with columns = frames,
        entries = 0/1 labels from each LLM annotator, index aligned with docs.
        Example:
            label_surrogates = {
                "claude": df_claude_val,  # columns: frames
                "gpt": df_gpt_val,
                "llama": df_llama_val,
            }

    Returns
    -------
    S : pd.DataFrame
        Surrogate matrix of shape (n_docs, n_surrogates_per_frame * len(frames)).
        Column naming convention:
            f"{frame}_{surrogate_name.lower()}"
    """
    # Determine number of docs
    n_docs = None

    if prob_surrogates:
        first_model = next(iter(prob_surrogates.values()))
        first_frame = frames[0]
        n_docs = len(first_model[first_frame])

    if n_docs is None and label_surrogates:
        first_llm_df = next(iter(label_surrogates.values()))
        n_docs = len(first_llm_df)

    if n_docs is None:
        raise ValueError("Unable to infer number of documents; provide prob_surrogates or label_surrogates with at least one source.")

    rows: List[Dict[str, Any]] = []
    for i in range(n_docs):
        row: Dict[str, Any] = {}
        for f in frames:
            # probabilistic surrogates
            if prob_surrogates:
                for name, per_frame_dict in prob_surrogates.items():
                    col_name = f"{f}_{name.lower()}"
                    val = per_frame_dict[f][i]
                    # make sure it's scalar
                    row[col_name] = float(val)
            # label-based surrogates (e.g., LLM labels)
            if label_surrogates:
                for name, df in label_surrogates.items():
                    col_name = f"{f}_{name.lower()}"
                    row[col_name] = int(df[f].iloc[i])
        rows.append(row)

    return pd.DataFrame(rows)


# -------------------------------------------------------------------------
# Meta-model fitting and prediction
# -------------------------------------------------------------------------

def _build_feature_matrices_for_frame(
    frame: str,
    S_labeled: pd.DataFrame,
    S_all: pd.DataFrame,
    X_labeled: Optional[pd.DataFrame] = None,
    X_all: Optional[pd.DataFrame] = None,
) -> Tuple[np.ndarray, np.ndarray, List[str], List[str]]:
    """
    Internal helper: build feature matrices for one frame.
    """
    # Choose surrogate columns that belong to this frame
    surrogate_cols = [c for c in S_labeled.columns if c.startswith(frame + "_")]

    if not surrogate_cols:
        raise ValueError(f"No surrogate columns found for frame '{frame}' in S_labeled.")

    if X_labeled is not None:
        extra_cols = list(X_labeled.columns)
        F_labeled = pd.concat([S_labeled[surrogate_cols], X_labeled], axis=1)
        F_all = pd.concat([S_all[surrogate_cols], X_all], axis=1)
    else:
        extra_cols = []
        F_labeled = S_labeled[surrogate_cols]
        F_all = S_all[surrogate_cols]

    return F_labeled.values, F_all.values, surrogate_cols, extra_cols


def fit_meta_for_frame(
    frame: str,
    S_labeled: pd.DataFrame,
    Y_labeled: pd.DataFrame,
    S_all: pd.DataFrame,
    X_labeled: Optional[pd.DataFrame] = None,
    X_all: Optional[pd.DataFrame] = None,
    C: float = 1.0,
) -> Tuple[DSLMetaModel, np.ndarray, np.ndarray]:
    """
    Fit a meta-model for a single frame using surrogate features and gold labels.

    Parameters
    ----------
    frame : str
        Frame name (column in Y_labeled).
    S_labeled : pd.DataFrame
        Surrogate matrix for labeled docs.
    Y_labeled : pd.DataFrame
        Gold labels for labeled docs (columns include `frame`).
    S_all : pd.DataFrame
        Surrogate matrix for all docs (labeled + unlabeled).
    X_labeled : pd.DataFrame, optional
        Extra features for labeled docs.
    X_all : pd.DataFrame, optional
        Extra features for all docs.
    C : float
        Inverse regularization strength for logistic regression.

    Returns
    -------
    meta_model : DSLMetaModel
        Fitted meta model container.
    p_all : np.ndarray
        Predicted probabilities for all docs (aligned with S_all).
    p_labeled : np.ndarray
        Predicted probabilities for labeled docs (aligned with Y_labeled).
    """
    F_labeled, F_all, surrogate_cols, extra_cols = _build_feature_matrices_for_frame(
        frame, S_labeled, S_all, X_labeled, X_all
    )

    y = Y_labeled[frame].values

    pipe = Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(max_iter=500, solver="lbfgs", C=C))
    ])

    pipe.fit(F_labeled, y)
    p_all = pipe.predict_proba(F_all)[:, 1]
    p_labeled = pipe.predict_proba(F_labeled)[:, 1]

    meta_model = DSLMetaModel(
        frame=frame,
        model=pipe,
        surrogate_cols=surrogate_cols,
        extra_feature_cols=extra_cols
    )

    return meta_model, p_all, p_labeled


# -------------------------------------------------------------------------
# DSL / AIPW estimator and bootstrap CIs
# -------------------------------------------------------------------------

def dsl_mean_estimate(
    p_all: np.ndarray,
    y_labeled: np.ndarray,
    p_labeled: np.ndarray,
    pi: float | np.ndarray,
) -> float:
    """
    Compute the DSL / AIPW estimate of mean Y over the target corpus.

    Parameters
    ----------
    p_all : np.ndarray
        Meta-model probabilities for all N documents.
    y_labeled : np.ndarray
        Gold labels for the labeled subset L (0/1), shape (n_L,).
    p_labeled : np.ndarray
        Meta-model probabilities for the labeled subset L (aligned with y_labeled).
    pi : float or np.ndarray
        Inclusion probability for each labeled document.
        If scalar, assumed same for all labeled docs (simple random sample).
        If array, shape (n_L,), possibly varying by stratum.

    Returns
    -------
    theta_hat : float
        DSL estimate of mean(Y) over the target corpus.
    """
    p_all = np.asarray(p_all, dtype=float)
    y_labeled = np.asarray(y_labeled, dtype=float)
    p_labeled = np.asarray(p_labeled, dtype=float)

    if np.isscalar(pi):
        pi_arr = np.full_like(y_labeled, fill_value=float(pi), dtype=float)
    else:
        pi_arr = np.asarray(pi, dtype=float)
        if pi_arr.shape != y_labeled.shape:
            raise ValueError("pi must have same shape as y_labeled if provided as array.")

    N = len(p_all)
    if N == 0:
        raise ValueError("p_all is empty; cannot compute DSL mean.")

    # AIPW / DSL estimator
    correction = np.mean((y_labeled - p_labeled) / pi_arr)
    theta_hat = float(p_all.mean() + correction)
    return theta_hat


def bootstrap_dsl_ci(
    p_all: np.ndarray,
    y_labeled: np.ndarray,
    p_labeled: np.ndarray,
    pi: float | np.ndarray,
    B: int = 500,
    alpha: float = 0.05,
    random_state: Optional[int] = None,
) -> Tuple[float, float, float]:
    """
    Bootstrap confidence interval for the DSL mean estimator (v2-style).

    This version:
      - resamples ALL docs for the mean(p_all) part
      - resamples labeled docs for the correction part

    Parameters
    ----------
    p_all : np.ndarray
        Meta-model probabilities for all N documents.
    y_labeled : np.ndarray
        Gold labels for labeled docs.
    p_labeled : np.ndarray
        Meta-model probabilities for labeled docs.
    pi : float or np.ndarray
        Inclusion probabilities for labeled docs.
    B : int
        Number of bootstrap resamples.
    alpha : float
        Nominal significance level for 1-alpha CI.
    random_state : int, optional
        Seed for reproducibility.

    Returns
    -------
    theta_hat : float
        Point estimate (original DSL mean).
    lower : float
        Lower CI bound.
    upper : float
        Upper CI bound.
    """
    rng = np.random.default_rng(random_state)

    p_all = np.asarray(p_all, dtype=float)
    y_labeled = np.asarray(y_labeled, dtype=float)
    p_labeled = np.asarray(p_labeled, dtype=float)

    N = len(p_all)
    n_L = len(y_labeled)
    if N == 0 or n_L == 0:
        raise ValueError("Need non-empty p_all and labeled data for bootstrap.")

    theta_hat = dsl_mean_estimate(p_all, y_labeled, p_labeled, pi)

    thetas = []
    for _ in range(B):
        # 1) Resample ALL docs for the mean(p_all) component
        idx_all = rng.integers(0, N, size=N)
        mean_all = float(p_all[idx_all].mean())

        # 2) Resample labeled docs for the correction component
        idx_L = rng.integers(0, n_L, size=n_L)
        y_b = y_labeled[idx_L]
        p_l_b = p_labeled[idx_L]

        if np.isscalar(pi):
            corr = np.mean((y_b - p_l_b) / float(pi))
        else:
            pi_arr = np.asarray(pi, dtype=float)
            if pi_arr.shape != y_labeled.shape:
                raise ValueError("pi must have same shape as y_labeled if provided as array.")
            corr = np.mean((y_b - p_l_b) / pi_arr[idx_L])

        thetas.append(mean_all + corr)

    thetas = np.asarray(thetas, dtype=float)
    lower, upper = np.quantile(thetas, [alpha / 2, 1 - alpha / 2])

    return float(theta_hat), float(lower), float(upper)



# -------------------------------------------------------------------------
# Threshold tuning for classification
# -------------------------------------------------------------------------

def tune_threshold(
    probs: np.ndarray,
    y_true: np.ndarray,
    metric: str = "f1",
    grid: Optional[np.ndarray] = None,
) -> Tuple[float, Dict[str, float]]:
    """
    Tune a classification threshold on probabilities to maximize a chosen metric.

    Parameters
    ----------
    probs : np.ndarray
        Ensemble probabilities (P(Y=1)).
    y_true : np.ndarray
        Gold labels (0/1).
    metric : str
        Currently supports "f1" (binary F1).
    grid : np.ndarray, optional
        Threshold grid to search. If None, use np.linspace(0.05, 0.95, 19).

    Returns
    -------
    best_t : float
        Best threshold on given grid.
    stats : dict
        Dictionary with best score, metric name, and maybe other info later.
    """
    probs = np.asarray(probs, dtype=float)
    y_true = np.asarray(y_true, dtype=int)

    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)

    best_t = 0.5
    best_score = -1.0

    for t in grid:
        pred = (probs >= t).astype(int)
        if metric == "f1":
            score = f1_score(y_true, pred, average="binary", zero_division=0)
        else:
            raise ValueError(f"Unsupported metric: {metric}")

        if score > best_score:
            best_score = score
            best_t = float(t)

    return best_t, {"metric": metric, "best_score": float(best_score)}


# -------------------------------------------------------------------------
# High-level helped for convenience: run DSL for all frames
# -------------------------------------------------------------------------

def run_dsl_for_all_frames(
    frames: List[str],
    S_labeled: pd.DataFrame,
    Y_labeled: pd.DataFrame,
    S_all: pd.DataFrame,
    pi_labeled: float | np.ndarray,
    X_labeled: Optional[pd.DataFrame] = None,
    X_all: Optional[pd.DataFrame] = None,
    do_threshold_tuning: bool = True,
    threshold_metric: str = "f1",
    do_bootstrap: bool = False,              
    bootstrap_B: int = 500,               
    bootstrap_alpha: float = 0.05,         
    bootstrap_random_state: Optional[int] = None,  
) -> Dict[str, DSLFrameResult]:
    """
    High-level helper to:
      - fit DSL meta-models for all frames
      - compute DSL mean prevalence estimates
      - (optionally) tune classification thresholds

    Parameters
    ----------
    frames : list of str
        Frame names (columns in Y_labeled).
    S_labeled : pd.DataFrame
        Surrogate matrix for labeled docs.
    Y_labeled : pd.DataFrame
        Gold labels for labeled docs.
    S_all : pd.DataFrame
        Surrogate matrix for all docs (labeled + unlabeled).
    pi_labeled : float or np.ndarray
        Inclusion probabilities for labeled docs.
    X_labeled : pd.DataFrame, optional
        Extra features for labeled docs.
    X_all : pd.DataFrame, optional
        Extra features for all docs.
    do_threshold_tuning : bool
        Whether to tune thresholds for classification.
    threshold_metric : str
        Metric for threshold tuning (currently "f1").

    Returns
    -------
    results : dict
        Mapping frame -> DSLFrameResult.
    """
    results: Dict[str, DSLFrameResult] = {}

    for frame in frames:
        meta_model, p_all, p_labeled = fit_meta_for_frame(
            frame=frame,
            S_labeled=S_labeled,
            Y_labeled=Y_labeled,
            S_all=S_all,
            X_labeled=X_labeled,
            X_all=X_all,
        )

        # Point estimate of mean prevalence
        theta_hat = dsl_mean_estimate(
            p_all=p_all,
            y_labeled=Y_labeled[frame].values,
            p_labeled=p_labeled,
            pi=pi_labeled,
        )


        # Optional bootstrap CIs (v2-style)
        ci_lower = ci_upper = None
        if do_bootstrap:
            theta_bs, ci_lower, ci_upper = bootstrap_dsl_ci(
                p_all=p_all,
                y_labeled=Y_labeled[frame].values,
                p_labeled=p_labeled,
                pi=pi_labeled,
                B=bootstrap_B,
                alpha=bootstrap_alpha,
                random_state=bootstrap_random_state,
            )
            # keep theta_hat as the same (theta_bs should be essentially equal)
            theta_hat = theta_bs
            
        # Optional threshold tuning (for classification use)
        threshold = None
        if do_threshold_tuning:
            threshold, _stats = tune_threshold(
                probs=p_labeled,
                y_true=Y_labeled[frame].values,
                metric=threshold_metric,
            )

        results[frame] = DSLFrameResult(
            frame=frame,
            theta_hat_mean_prevalence=float(theta_hat),
            meta_model=meta_model,
            p_all=p_all,
            p_labeled=p_labeled,
            threshold=threshold,
            ci_lower=ci_lower,
            ci_upper=ci_upper,
        )

    return results

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split

SAVE_DIR = "all_run3"   # same as in framing_training.py
N_TRAIN = 240                    # MUST match the run used to create the pickles
OVERLAP_TRAIN_VAL = False        # MUST match framing_training split logic

# Paths to core data
TRAIN_CSV = "labelled_frames_train.csv"
TEST_CSV  = "labelled_frames_test.csv"

# Paths to LLM annotations (0/1 per frame, same row order as val/test splits)
claude_VAL_CSV  = "/data_users1/rush17m/computational-framing/Framing/LLM/Predicted/All_preds/claude_zero-shot.csv"
gpt_VAL_CSV  = "/data_users1/rush17m/computational-framing/Framing/LLM/Predicted/All_preds/gpt-oss_zero-shot.csv"
llama_VAL_CSV  = "/data_users1/rush17m/computational-framing/Framing/LLM/Predicted/All_preds/llama_zero-shot.csv"
claude_TEST_CSV = "/data_users1/rush17m/computational-framing/Framing/LLM/Predicted/Test/claude-sonnet-4-20250514_zshot.csv"
gpt_TEST_CSV = "/data_users1/rush17m/computational-framing/Framing/LLM/Predicted/Test/gpt-oss-20b-zshot.csv"
llama_TEST_CSV = "/data_users1/rush17m/computational-framing/Framing/LLM/Predicted/Test/llama3.1:8b-instruct-q8_0_zshot.csv"


# ---------------------------------------------------------------------
# 1. Load base data and reproduce the val/test split
# ---------------------------------------------------------------------

df_train_full = pd.read_csv(TRAIN_CSV)
df_test_full  = pd.read_csv(TEST_CSV)

frames = list(df_test_full.columns[-7:])   # assuming last 7 cols are frame labels
cols_for_x = ["title", "text"]

if N_TRAIN < len(df_train_full) and not OVERLAP_TRAIN_VAL:
    # same as in framing_training.py
    X_train_BERT, X_val_BERT, y_train, y_val = train_test_split(
        df_train_full[cols_for_x],
        df_train_full[frames],
        test_size=1.0 - (N_TRAIN / len(df_train_full)),
        random_state=5,
    )
elif N_TRAIN == len(df_train_full):
    X_train_BERT = df_train_full[cols_for_x]
    X_val_BERT   = df_train_full[cols_for_x]
    y_train      = df_train_full[frames]
    y_val        = df_train_full[frames]
else:
    X_train_BERT, _, y_train, _ = train_test_split(
        df_train_full[cols_for_x],
        df_train_full[frames],
        test_size=1.0 - (N_TRAIN / len(df_train_full)),
        random_state=5,
    )
    X_val_BERT, _, y_val, _ = train_test_split(
        df_train_full[cols_for_x],
        df_train_full[frames],
        test_size=1.0 - (N_TRAIN / len(df_train_full)),
        random_state=42,
    )

# Test split is just the test file
X_test_BERT = df_test_full[cols_for_x]
y_test      = df_test_full[frames]

n_val  = len(X_val_BERT)
n_test = len(X_test_BERT)

print(f"[INFO] Frames: {frames}")
print(f"[INFO] Val size: {n_val}, Test size: {n_test}")


# ---------------------------------------------------------------------
# 2. Load NB/BERT/DeBERTa stats_and_preds pickles
# ---------------------------------------------------------------------

nb_res       = pickle.load(open(os.path.join(SAVE_DIR, "NB_stats_and_preds.pkl"), "rb"))
deberta_res  = pickle.load(open(os.path.join(SAVE_DIR, "DeBERTa_stats_and_preds.pkl"), "rb"))
bert_res  = pickle.load(open(os.path.join(SAVE_DIR, "BERT_stats_and_preds.pkl"), "rb"))
# bert_path = os.path.join(SAVE_DIR, "BERT_stats_and_preds.pkl")
# bert_res = pickle.load(open(bert_path, "rb")) if os.path.exists(bert_path) else None

print("[INFO] Loaded NB / DeBERTa", "and BERT" if bert_res else "(no BERT found)")

# Helper to concat val + test probs for one model
def concat_val_test(model_res, frame: str):
    val_probs  = np.asarray(model_res["val_continuous"][frame],  dtype=float)
    test_probs = np.asarray(model_res["test_continuous"][frame], dtype=float)
    return val_probs, test_probs, np.concatenate([val_probs, test_probs], axis=0)

def load_llm_test_labels(csv_path: str,
                         df_test: pd.DataFrame,
                         frames: list[str]) -> pd.DataFrame:
    """
    Load LLM labels for TEST and align rows with df_test using stories_id.
    Returns a DataFrame with only the frame columns, in the same row order
    as df_test. Any missing labels are filled with 0.
    """
    df_llm = pd.read_csv(csv_path)

    needed_cols = ["stories_id"] + frames
    missing_cols = [c for c in needed_cols if c not in df_llm.columns]
    if missing_cols:
        raise ValueError(f"{csv_path} is missing columns: {missing_cols}")

    df_llm = df_llm[needed_cols]

    df_merged = (
        df_test[["stories_id"]]
        .merge(
            df_llm,
            on="stories_id",
            how="left",
            validate="one_to_one",
        )
    )

    # Warn but don't die if there are NaNs
    if df_merged[frames].isna().any().any():
        na_rows = df_merged[df_merged[frames].isna().any(axis=1)]
        print(f"[WARN] {csv_path}: {na_rows.shape[0]} test rows have NaN LLM labels.")
        print("Example problematic stories_id:",
              na_rows["stories_id"].head().tolist())
        # Fill missing label with 0
        df_merged[frames] = df_merged[frames].fillna(0)

    # Force int 0/1
    df_labels = df_merged[frames].astype(int).reset_index(drop=True)
    return df_labels


# ---------------------------------------------------------------------
# 3. Load LLM 0/1 labels for val and test
#    (These CSVs to have columns = frames, rows aligned with val/test.)
# ---------------------------------------------------------------------

df_claude  = pd.read_csv(claude_VAL_CSV)
df_gpt  = pd.read_csv(gpt_VAL_CSV)
df_llama  = pd.read_csv(llama_VAL_CSV)

claude_map = df_claude.set_index("stories_id")[frames]
gpt_map = df_gpt.set_index("stories_id")[frames]
llama_map = df_llama.set_index("stories_id")[frames]

# use stories_id from val rows to index into that map
stories_val = df_train_full.loc[X_val_BERT.index, "stories_id"]

df_claude_val = (
    pd.DataFrame({"stories_id": stories_val})
    .merge(df_claude[["stories_id"] + frames],on="stories_id",how="left",validate="one_to_one",)
    [frames]
    .reset_index(drop=True)
)
df_gpt_val = (
    pd.DataFrame({"stories_id": stories_val})
    .merge(df_gpt[["stories_id"] + frames],on="stories_id",how="left",validate="one_to_one",)
    [frames]
    .reset_index(drop=True)
)
df_llama_val = (
    pd.DataFrame({"stories_id": stories_val})
    .merge(df_llama[["stories_id"] + frames],on="stories_id",how="left",validate="one_to_one",)
    [frames]
    .reset_index(drop=True)
)
df_claude_val = df_claude_val.fillna(0)
df_gpt_val = df_gpt_val.fillna(0)
df_llama_val = df_llama_val.fillna(0)


df_claude_test = load_llm_test_labels(claude_TEST_CSV, df_test_full, frames)
df_gpt_test    = load_llm_test_labels(gpt_TEST_CSV,    df_test_full, frames)
df_llama_test  = load_llm_test_labels(llama_TEST_CSV,  df_test_full, frames)

# Sanity checks
assert list(df_claude_val.columns)  == frames, "claude val columns must equal `frames`"
assert list(df_claude_test.columns) == frames, "claude test columns must equal `frames`"

# Build ALL (val + test) LLM label frames
df_claude_all = pd.concat([df_claude_val, df_claude_test], axis=0).reset_index(drop=True)
df_gpt_all = pd.concat([df_gpt_val, df_gpt_test], axis=0).reset_index(drop=True)
df_llama_all = pd.concat([df_llama_val, df_llama_test], axis=0).reset_index(drop=True)


# ---------------------------------------------------------------------
# 4. Build surrogate matrices S_val and S_all
# ---------------------------------------------------------------------

# 4.1 Probabilistic surrogates for VAL only
prob_surrogates_val = {
    "NB":      {f: nb_res["val_continuous"][f]      for f in frames},
    "DeBERTa": {f: deberta_res["val_continuous"][f] for f in frames},
}
if bert_res:
    prob_surrogates_val["BERT"] = {f: bert_res["val_continuous"][f] for f in frames}

# 4.2 Probabilistic surrogates for ALL (val + test)
prob_surrogates_all = {
    "NB":      {},
    "DeBERTa": {},
}
if bert_res:
    prob_surrogates_all["BERT"] = {}

for f in frames:
    # NB
    val_nb, test_nb, all_nb = concat_val_test(nb_res, f)
    prob_surrogates_all["NB"][f] = all_nb

    # DeBERTa
    val_deb, test_deb, all_deb = concat_val_test(deberta_res, f)
    prob_surrogates_all["DeBERTa"][f] = all_deb

    # BERT (if available)
    if bert_res:
        val_b, test_b, all_b = concat_val_test(bert_res, f)
        prob_surrogates_all["BERT"][f] = all_b

# 4.3 Label surrogates (LLM) for VAL and ALL
label_surrogates_val = {
    "claude": df_claude_val.reset_index(drop=True),
    "gpt": df_gpt_val.reset_index(drop=True),
    "llama": df_llama_val.reset_index(drop=True),
}
label_surrogates_all = {
    "claude": df_claude_all,
    "gpt": df_gpt_all,
    "llama": df_llama_all,
}

# 4.4 Build S_val and S_all using dsl_ensemble utilities

S_val = build_surrogate_df(
    frames=frames,
    prob_surrogates=prob_surrogates_val,
    label_surrogates=label_surrogates_val,
)
S_all = build_surrogate_df(
    frames=frames,
    prob_surrogates=prob_surrogates_all,
    label_surrogates=label_surrogates_all,
)

print(f"[INFO] S_val shape: {S_val.shape}, S_all shape: {S_all.shape}")

# ---------------------------------------------------------------------
# 5. Run DSL for all frames
# ---------------------------------------------------------------------

# Gold labels for labeled set
Y_labeled = y_val.reset_index(drop=True)

# Inclusion probability: if val is a simple random subset of val+test:
pi_labeled = len(S_val) / len(S_all)
print(f"[INFO] Using pi_labeled = {pi_labeled:.4f} (SRS assumption)")

dsl_results = run_dsl_for_all_frames(
    frames=frames,
    S_labeled=S_val,
    Y_labeled=Y_labeled,
    S_all=S_all,
    pi_labeled=pi_labeled,
    X_labeled=None,
    X_all=None,
    do_threshold_tuning=True,
    threshold_metric="f1",
    do_bootstrap=True,  
    bootstrap_B=500,
    bootstrap_alpha=0.05,
    bootstrap_random_state=42,
)

for f, r in dsl_results.items():
    print(
        f,
        "theta_hat=",
        r.theta_hat_mean_prevalence,
        "CI=[",
        r.ci_lower,
        ",",
        r.ci_upper,
        "]",
        "threshold=",
        r.threshold,
    )


# ---------------------------------------------------------------------
# 6. Evaluate ensemble on TEST set using DSL thresholds
# ---------------------------------------------------------------------

print("\n=== TEST-SET EVALUATION USING DSL ENSEMBLE ===\n")

# p_all for each frame is for [val; test] in that row order.
# So we slice off the last n_test entries for test.
for f in frames:
    res_f = dsl_results[f]
    p_all = res_f.p_all
    assert len(p_all) == n_val + n_test, "S_all / p_all length mismatch."

    p_test = p_all[n_val:]  # last n_test docs are test
    t = res_f.threshold if res_f.threshold is not None else 0.5

    y_test_f = y_test[f].values
    y_pred_test = (p_test >= t).astype(int)

    f1 = f1_score(y_test_f, y_pred_test, average="binary", zero_division=0)
    print(f"[{f}] threshold={t:.2f}, test F1={f1:.3f}")

print("\nDetailed classification report for one example frame:\n")
example_frame = frames[0]
res_ex = dsl_results[example_frame]
p_all_ex = res_ex.p_all
p_test_ex = p_all_ex[n_val:]
y_test_ex = y_test[example_frame].values
y_pred_ex = (p_test_ex >= res_ex.threshold).astype(int)

print(f"=== Frame: {example_frame} ===")
print(classification_report(y_test_ex, y_pred_ex, digits=3))


[INFO] Frames: ['sexual stigma and transmission routes', 'racial disparities and stigmatising name', 'global relations', 'public health failure', 'epidemic preparedness and surveillance', 'human-interest stories', 'broader health issues']
[INFO] Val size: 160, Test size: 100
[INFO] Loaded NB / DeBERTa and BERT
[WARN] /data_users1/rush17m/computational-framing/Framing/LLM/Predicted/Test/gpt-oss-20b-zshot.csv: 3 test rows have NaN LLM labels.
Example problematic stories_id: [2385295037, 2374724950, 2408721783]
[INFO] S_val shape: (160, 42), S_all shape: (260, 42)
[INFO] Using pi_labeled = 0.6154 (SRS assumption)
sexual stigma and transmission routes theta_hat= 0.37386938594604974 CI=[ 0.27374274751988925 , 0.47177200134662267 ] threshold= 0.39999999999999997
racial disparities and stigmatising name theta_hat= 0.08699257178550265 CI=[ 0.04206947994232553 , 0.1322235347549885 ] threshold= 0.35
global relations theta_hat= 0.04355163463361362 CI=[ -0.006779864827422183 , 0.09621042077990469 

In [ ]:
import os
import numpy as np
import pandas as pd

# Base columns to preserve from the test file
base_cols = [
    "stories_id",
    "media_name",
    "title",
    "url",
    "text",
    "keywords",
]

# Start from the original test dataframe, keep only those columns
df_dsl_test = df_test_full[base_cols].copy()

# For each frame, slice out the TEST probabilities from p_all, apply threshold,
# and store 0/1 predictions in the corresponding column.
for f in frames:
    res_f = dsl_results[f]
    # p_all is for [VAL; TEST] in that order
    p_all = res_f.p_all
    assert len(p_all) == n_val + n_test, f"Length mismatch for frame {f}"
    # last n_test entries correspond to the TEST split
    p_test = p_all[n_val:] # shape: (n_test,)
    t = res_f.threshold if res_f.threshold is not None else 0.5

    # 0/1 predictions
    y_pred_test = (p_test >= t).astype(int)
    df_dsl_test[f] = y_pred_test


# Save to CSV in run directory
out_path = os.path.join(SAVE_DIR, "dsl_test_predictions3.csv")
df_dsl_test.to_csv(out_path, index=False)

print(f"[INFO] Saved DSL test predictions to: {out_path}")
print(df_dsl_test.head(3))


[INFO] Saved DSL test predictions to: all_run3/dsl_test_predictions3.csv
   stories_id         media_name  \
0  2312961630           Fox News   
1  2312977252  NBC Breaking News   
2  2315055196          Breitbart   

                                               title  \
0  Massachusetts confirms first case of monkeypox...   
1  Monkeypox case identified in Massachusetts as ...   
2  W.H.O.: Nigeria at Risk of 'Ongoing Transmissi...   

                                                 url  \
0  https://www.foxnews.com/health/massachusetts-c...   
1  https://www.nbcnews.com/health/health-news/mon...   
2  https://www.breitbart.com/africa/2022/05/20/w-...   

                                                text  \
0  Massachusetts health officials confirmed on We...   
1  A man in Massachusetts marks the first U.S. ca...   
2  The World Health Organization (W.H.O.) warned ...   

                                            keywords  \
0  ['confirms', 'monkeypox', 'massachusetts', 'ca..